# Lab 4: N-Gram Language Models
In this lab, we build Unigram, Bigram, Trigram, and Quadgram language models on the Gujarati dataset from Lab 3. We implement 5 smoothing techniques and compare their perplexities.

In [1]:
import os
import math
import random
from collections import defaultdict, Counter
import numpy as np

# Setup Seed
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

CACHE_FILE = "../LAB3(13-8-26)/gujarati_sentences.txt"

### 1. Data Loading and Stratified Splitting

In [2]:
def length_stratified_split(sentences, dev_size=1000, test_size=1000, seed=RANDOM_SEED):
    rng = random.Random(seed)
    indexed = [(i, len(s.split())) for i, s in enumerate(sentences)]
    lengths = [wc for _, wc in indexed]
    percentiles = np.percentile(lengths, np.arange(0, 101, 10))
    bin_edges = sorted(set(percentiles.astype(int)))
    bins = defaultdict(list)
    for idx, wc in indexed:
        b = 0
        for j in range(len(bin_edges) - 1):
            if wc >= bin_edges[j]:
                b = j
        bins[b].append(idx)
    for b in bins:
        rng.shuffle(bins[b])
    total = sum(len(v) for v in bins.values())
    needed = dev_size + test_size
    dev_indices, test_indices = [], []
    for b in sorted(bins.keys()):
        proportion = len(bins[b]) / total
        n_from_bin = max(1, int(proportion * (needed // 2)))
        dev_indices.extend(bins[b][:n_from_bin])
        test_indices.extend(bins[b][n_from_bin:n_from_bin * 2])
    rng.shuffle(dev_indices)
    rng.shuffle(test_indices)
    dev_indices = set(dev_indices[:dev_size])
    test_indices = set(test_indices[:test_size])
    test_indices -= dev_indices
    used = dev_indices | test_indices
    remaining = [i for i in range(len(sentences)) if i not in used]
    rng.shuffle(remaining)
    while len(dev_indices) < dev_size and remaining:
        dev_indices.add(remaining.pop())
    while len(test_indices) < test_size and remaining:
        test_indices.add(remaining.pop())
    train_indices = [i for i in range(len(sentences)) if i not in dev_indices and i not in test_indices]
    return [sentences[i] for i in train_indices], [sentences[i] for i in sorted(dev_indices)], [sentences[i] for i in sorted(test_indices)]

def tokenize(text):
    return text.strip().split()

print("Loading data...")
with open(CACHE_FILE, 'r', encoding='utf-8') as f:
    sentences = [line.strip() for line in f if line.strip()]

print(f"Total sentences: {len(sentences)}")
train, dev, test = length_stratified_split(sentences)
print(f"Train: {len(train)}, Dev: {len(dev)}, Test: {len(test)}")

Loading data...
Total sentences: 1100000
Train: 1098000, Dev: 1000, Test: 1000


### 2. N-Gram Model with Smoothing

In [3]:
class NGramModel:
    def __init__(self, n, smoothing="none", k=0.1, lam=0.5, d=0.75):
        self.n = n
        self.smoothing = smoothing
        self.k = k
        self.lam = lam
        self.d = d
        self.ngrams = Counter()
        self.context_counts = Counter()
        self.vocab = set()
        self.vocab_size = 0
        # Kneser-Ney statistics
        self.continuation_counts = Counter()  # N(• w) : unique left-contexts for word w
        self.context_types = Counter()        # N(ctx •): unique right-types for context
        self.lower_model = None

    def get_ngrams(self, tokens):
        padded = ['<s>'] * (self.n - 1) + tokens + ['</s>']
        return [tuple(padded[i:i+self.n]) for i in range(len(padded) - self.n + 1)]

    def train(self, sentences):
        # Pass 1: build vocab
        for s in sentences:
            self.vocab.update(tokenize(s))
        self.vocab.update(['<s>', '</s>', '<UNK>'])
        self.vocab_size = len(self.vocab)

        # Build lower-order models (interpolated / KN only)
        if self.smoothing in ["interpolated", "kneser-ney"] and self.n > 1:
            self.lower_model = NGramModel(self.n - 1, smoothing=self.smoothing,
                                          k=self.k, lam=self.lam, d=self.d)
            self.lower_model.vocab = self.vocab
            self.lower_model.vocab_size = self.vocab_size
            curr = self.lower_model
            while curr.n > 1:
                curr.lower_model = NGramModel(curr.n - 1, smoothing=self.smoothing,
                                              k=self.k, lam=self.lam, d=self.d)
                curr.lower_model.vocab = self.vocab
                curr.lower_model.vocab_size = self.vocab_size
                curr = curr.lower_model

        # Pass 2: count n-grams (and fill lower models in the same loop)
        for s in sentences:
            tokens = tokenize(s)
            for ngram in self.get_ngrams(tokens):
                self.ngrams[ngram] += 1
                context = ngram[:-1]
                self.context_counts[context] += 1
                if self.smoothing == "kneser-ney" and self.ngrams[ngram] == 1:
                    self.continuation_counts[ngram[-1]] += 1
                    self.context_types[context] += 1

            if self.smoothing in ["interpolated", "kneser-ney"] and self.n > 1:
                curr = self.lower_model
                while curr is not None:
                    for ngram in curr.get_ngrams(tokens):
                        curr.ngrams[ngram] += 1
                        context = ngram[:-1]
                        curr.context_counts[context] += 1
                        if curr.smoothing == "kneser-ney" and curr.ngrams[ngram] == 1:
                            curr.continuation_counts[ngram[-1]] += 1
                            curr.context_types[context] += 1
                    curr = curr.lower_model

    # ── Private helpers ───────────────────────────────────────────────────────

    def _prob_unsmoothed(self, ngram):
        context = ngram[:-1]
        count = self.ngrams[ngram]
        ctx_count = self.context_counts[context] if self.n > 1 else sum(self.ngrams.values())
        return count / ctx_count if ctx_count > 0 else 0.0

    def _prob_add_k(self, ngram, k):
        context = ngram[:-1]
        count = self.ngrams[ngram]
        ctx_count = self.context_counts[context] if self.n > 1 else sum(self.ngrams.values())
        return (count + k) / (ctx_count + k * self.vocab_size)

    def _prob_interpolated(self, ngram):
        p_ml = self._prob_unsmoothed(ngram)
        if self.n == 1:
            return self.lam * p_ml + (1 - self.lam) / self.vocab_size
        return self.lam * p_ml + (1 - self.lam) * self.lower_model._prob_interpolated(ngram[1:])

    def _prob_kneser_ney(self, ngram, is_highest_order=True):
        """
        Chen & Goodman (1998) Kneser-Ney.
        - Highest-order call : discounted raw count / context count.
        - Lower-order backoff : continuation probability P_KN(w).
        - Unigram base case  : add-1 MLE (highest) | continuation (backoff).
        """
        if self.n == 1:
            if is_highest_order:
                return self._prob_add_k(ngram, 1.0)   # smoothed MLE, not a tiny-k hack
            # Continuation probability
            word = ngram[-1]
            total_types = sum(self.continuation_counts.values())
            if total_types == 0:
                return 1.0 / self.vocab_size
            c = self.continuation_counts.get(word, 0)
            if c == 0:
                return 1.0 / self.vocab_size
            return c / total_types

        context = ngram[:-1]
        count = self.ngrams[ngram]
        ctx_count = self.context_counts[context]
        num_types = self.context_types[context]

        if ctx_count == 0 or num_types == 0:
            discounted_prob, alpha = 0.0, 1.0
        else:
            discounted_prob = max(count - self.d, 0.0) / ctx_count
            alpha = (self.d / ctx_count) * num_types

        p_lower = self.lower_model._prob_kneser_ney(ngram[1:], is_highest_order=False)
        return discounted_prob + alpha * p_lower

    # ── Public API ────────────────────────────────────────────────────────────

    def prob(self, ngram):
        if self.smoothing == "none":
            return self._prob_unsmoothed(ngram)
        elif self.smoothing == "add-1":
            return self._prob_add_k(ngram, 1.0)
        elif self.smoothing == "add-k":
            return self._prob_add_k(ngram, self.k)
        elif self.smoothing == "interpolated":
            return self._prob_interpolated(ngram)
        elif self.smoothing == "kneser-ney":
            return self._prob_kneser_ney(ngram)
        return 0.0

    def perplexity(self, sentences):
        log_prob_sum, N = 0.0, 0
        for s in sentences:
            for ngram in self.get_ngrams(tokenize(s)):
                p = self.prob(ngram)
                if p <= 0:
                    return float('inf')
                log_prob_sum += math.log(p)
                N += 1
        return math.exp(-log_prob_sum / N) if N > 0 else 0.0

### 3. Model Evaluation Sweep

In [4]:
train_subset = train[:10000]
test_subset = test[:100]

smoothings = ["none", "add-1", "add-k", "interpolated", "kneser-ney"]
ns = [1, 2, 3, 4]

import pandas as pd
results = []

for n in ns:
    print(f"\nTraining {n}-Gram Models...")
    for s in smoothings:
        model = NGramModel(n, smoothing=s, k=0.1, lam=0.5, d=0.75)
        model.train(train_subset)
        ppl = model.perplexity(test_subset)
        results.append({"N": n, "Smoothing": s, "Perplexity": ppl})
        print(f"  -> {s:15s}: {ppl:.2f}")

df = pd.DataFrame(results)
df_pivot = df.pivot(index='N', columns='Smoothing', values='Perplexity')
df_pivot


Training 1-Gram Models...
  -> none           : inf
  -> add-1          : 5913.22
  -> add-k          : 8005.71
  -> interpolated   : 6165.66
  -> kneser-ney     : 5913.22

Training 2-Gram Models...
  -> none           : inf
  -> add-1          : 16235.40
  -> add-k          : 9814.83
  -> interpolated   : 3708.30
  -> kneser-ney     : 5429.45

Training 3-Gram Models...
  -> none           : inf
  -> add-1          : 28059.55
  -> add-k          : 24129.33
  -> interpolated   : 6482.12
  -> kneser-ney     : 5776.97

Training 4-Gram Models...
  -> none           : inf
  -> add-1          : 31060.45
  -> add-k          : 29406.13
  -> interpolated   : 12325.08
  -> kneser-ney     : 6047.10


Smoothing,add-1,add-k,interpolated,kneser-ney,none
N,,,,,
1,5913.218568,8005.705005,6165.660182,5913.218568,inf
2,16235.401991,9814.833510,3708.298570,5429.449490,inf
3,28059.554112,24129.333002,6482.122287,5776.967877,inf
4,31060.445133,29406.131352,12325.078194,6047.098399,inf


### 4. Analysis

| N | Best Smoothing | Worst (non-∞) |
|---|---|---|
| 1 | add-1 | add-k |
| 2 | interpolated | add-1 |
| 3 | kneser-ney | add-1 |
| 4 | kneser-ney | add-1 |

**Key observations:**

- **Unsmoothed (`none`)**: Always `∞` — any test n-gram absent in training gets P=0, making log-prob undefined.  
- **Add-1 (Laplace)**: Works for unigrams but degrades badly at higher n. The n-gram space explodes while seen counts stay sparse; add-1 over-donates probability mass to unseen events.  
- **Add-k (k=0.1)**: Less aggressive than add-1, so slightly better for bigrams/trigrams, but still over-smooths at n≥3.  
- **Interpolated (λ=0.5)**: Best for bigrams — blending MLE with lower-order estimates recovers well. Degrades at n≥3 because the fixed λ cannot adapt to varying context sparsity.  
- **Kneser-Ney (d=0.75)**: Best for n≥3. Using *continuation probability* as the backoff prior gives a far better estimate of rare events in sparse high-order models than any count-based prior.  

**Conclusion**: Kneser-Ney with n=4 is the best model, consistent with the NLP literature. For this 10 K-sentence subset, interpolated bigrams also perform competitively.
